# Day 5 — Python → C++ / Rust Code Generator

Extends Day 4 to support **Rust** as a target language.

New in Day 5:
- AI-assisted Rust toolchain setup (the LLM reads your system info and advises you).
- Language selector: switch between C++ and Rust.
- Rust compile flags tuned for maximum runtime performance.
- Full Gradio UI matching the Day 4 experience.

## Quick start
1. Run the **Prerequisite Check** cell.
2. Fill in `.env` (GROQ_API_KEY, OPENROUTER_API_KEY; GOOGLE_API_KEY is optional).
3. Run cells top-to-bottom.


In [ ]:
# ── PREREQUISITE CHECK ───────────────────────────────────────────────────────
import shutil, importlib
from pathlib import Path

ok = True

for pkg, cmd in {
    "openai":  "pip install openai",
    "dotenv":  "pip install python-dotenv",
    "gradio":  "pip install gradio",
    "IPython": "pip install ipython",
}.items():
    try:
        importlib.import_module(pkg)
        print(f"✅ {pkg}")
    except ImportError:
        print(f"❌ Missing: '{pkg}'  →  {cmd}")
        ok = False

# C++ compiler (optional if using Rust only)
cpp_ok = any(shutil.which(c) for c in ("clang++", "g++"))
print("✅ C++ compiler found" if cpp_ok else "⚠️  No C++ compiler (needed for C++ target)")

# Rust toolchain (optional if using C++ only)
rust_ok = bool(shutil.which("rustc"))
print("✅ rustc found" if rust_ok else "⚠️  rustc not found (needed for Rust target)")
if not rust_ok:
    print("   Install: winget install Rustlang.Rustup  (Windows)")
    print("         or: curl https://sh.rustup.rs | sh  (Linux/macOS)")



✅ openai
✅ dotenv
✅ gradio
✅ IPython
✅ C++ compiler found
✅ rustc found


In [ ]:
# ── IMPORTS ──────────────────────────────────────────────────────────────────
import sys, os, platform
sys.path.insert(0, ".")

import gradio as gr
from IPython.display import Markdown, display

from src.code_utils import run_python, write_output
from src.compiler   import (
    pick_cpp_compiler, get_cpp_commands, compile_and_run_cpp,
    get_rust_commands,  compile_and_run_rust,
)
from src.llm_client import build_clients, call_llm, check_required_keys
from system_info    import retrieve_system_info, rust_toolchain_info
from styles         import CSS

print("✅ Imports OK")


✅ Imports OK


In [ ]:
# ── API KEY LOADING ──────────────────────────────────────────────────────────
# GOOGLE_API_KEY is optional — Gemini is disabled if absent.
from dotenv import load_dotenv
load_dotenv(override=True)

groq_api_key       = os.getenv("GROQ_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
google_api_key     = os.getenv("GOOGLE_API_KEY")  # optional

missing = [k for k, v in {
    "GROQ_API_KEY": groq_api_key,
    "OPENROUTER_API_KEY": openrouter_api_key,
}.items() if not v]

if missing:
    raise EnvironmentError(
        f"Missing required API key(s): {', '.join(missing)}\n"
        "Add them to .env and restart the kernel."
    )

print(f"✅ GROQ_API_KEY      : ...{groq_api_key[-6:]}")
print(f"✅ OPENROUTER_API_KEY: ...{openrouter_api_key[-6:]}")
if google_api_key:
    print(f"✅ GOOGLE_API_KEY    : ...{google_api_key[-6:]}  (Gemini enabled)")
else:
    print("ℹ️  GOOGLE_API_KEY not set — Gemini will be skipped")


✅ GROQ_API_KEY      : ...Md5wA9
✅ OPENROUTER_API_KEY: ...464067
✅ GOOGLE_API_KEY    : ...HYoZNE  (Gemini enabled)


In [ ]:
# ── LLM CLIENTS ──────────────────────────────────────────────────────────────
_clients   = build_clients()
groq       = _clients["groq"]
openrouter = _clients["openrouter"]
gemini     = _clients["gemini"]   # None if GOOGLE_API_KEY not set

# BUG FIX: openai/gpt-oss-120b is an OpenRouter model, NOT a Groq model.
# Groq does not serve that model ID; using it on groq raises a 404.
models = ["qwen/qwen3-coder-30b-a3b-instruct", "qwen/qwen3-32b"]
clients = {
    "qwen/qwen3-coder-30b-a3b-instruct": openrouter,
    "qwen/qwen3-32b":                    openrouter,
}

# Add Gemini only if the key is available
if gemini:
    models.insert(0, "gemini-2.5-flash")
    clients["gemini-2.5-flash"] = gemini

# Add a Groq model for fast, cheap iterations
models.append("llama-3.3-70b-versatile")
clients["llama-3.3-70b-versatile"] = groq

print("✅ Clients initialized")
for m in models:
    provider = next(k for k, v in _clients.items() if v is clients[m])
    print(f"   {m}  [{provider}]")


✅ Clients initialized
   gemini-2.5-flash  [gemini]
   qwen/qwen3-coder-30b-a3b-instruct  [openrouter]
   qwen/qwen3-32b  [openrouter]
   llama-3.3-70b-versatile  [groq]


In [ ]:
# ── SYSTEM & RUST TOOLCHAIN INFO ─────────────────────────────────────────────
system_info = retrieve_system_info()
rust_info   = rust_toolchain_info()
print("✅ System info collected")
print(f"   Rust installed: {rust_info['installed']}")
if rust_info["installed"]:
    print(f"   rustc version : {rust_info['rustc']['version']}")
rust_info


✅ System info collected
   Rust installed: True
   rustc version : rustc 1.95.0 (59807616e 2026-04-14)


{'installed': True,
 'rustc': {'path': 'C:\\Users\\HP\\.cargo\\bin\\rustc.EXE',
  'version': 'rustc 1.95.0 (59807616e 2026-04-14)',
  'host_triple': 'x86_64-pc-windows-msvc',
  'release': '1.95.0',
  'commit_hash': '59807616e1fa2540724bfbac14d7976d7e4a3860'},
 'cargo': {'path': 'C:\\Users\\HP\\.cargo\\bin\\cargo.EXE',
  'version': 'cargo 1.95.0 (f2d3ce0bd 2026-03-21)'},
 'rustup': {'path': 'C:\\Users\\HP\\.cargo\\bin\\rustup.EXE',
  'version': 'rustup 1.29.0 (28d1352db 2026-03-05)',
  'active_toolchain': 'stable-x86_64-pc-windows-msvc (default)',
  'default_toolchain': '',
  'toolchains': ['stable-x86_64-pc-windows-msvc (active, default)'],
  'targets_installed': ['x86_64-pc-windows-msvc']},
 'rust_analyzer': {'path': 'C:\\Users\\HP\\.cargo\\bin\\rust-analyzer.EXE'},
 'env': {'CARGO_HOME': 'C:\\Users\\HP\\.cargo',
  'RUSTUP_HOME': 'C:\\Users\\HP\\.rustup',
  'RUSTFLAGS': '',
  'CARGO_BUILD_TARGET': ''},
 'execution_examples': ['"C:\\Users\\HP\\.cargo\\bin\\cargo.EXE" build',
  '"C:\\Us

In [ ]:
# ── AI RUST TOOLCHAIN ADVISOR ────────────────────────────────────────────────
# Ask the LLM for Rust install instructions or the best compile flags
# for this machine.  Skip this cell if Rust is already installed.

if rust_info["installed"]:
    print("✅ Rust already installed — skipping advisor cell.")
else:
    advisor_prompt = f"""
Here is the system information for my computer.
I want to compile a single Rust file (main.rs) and execute it.

Please tell me:
1. Do I need to install a Rust toolchain?  If so, the simplest step-by-step instructions.
2. Once installed, what are the exact compile and run commands in Python subprocess form
   that give the FASTEST possible runtime performance on this machine?

Reply in Markdown.

System information:
{system_info}

Rust toolchain information:
{rust_info}
"""
    response = openrouter.chat.completions.create(
        model="qwen/qwen3-32b",
        messages=[{"role": "user", "content": advisor_prompt}],
    )
    display(Markdown(response.choices[0].message.content))


✅ Rust already installed — skipping advisor cell.


In [ ]:
# ── COMPILER COMMAND SETUP ───────────────────────────────────────────────────
# Language switch: change to "C++" to generate C++ instead.
LANGUAGE = "Rust"   # or "C++"

if LANGUAGE == "Rust":
    rustc_path = rust_info["rustc"]["path"] or "rustc"
    if not rustc_path:
        print("⚠️  rustc not found — run the advisor cell above for install instructions.")
        rustc_path = "rustc"   # will fail gracefully at compile time
    compile_command, run_command = get_rust_commands(rustc_path, "main.rs")
    source_file = "main.rs"
else:
    compiler = pick_cpp_compiler()
    compile_command, run_command = get_cpp_commands(compiler)
    source_file = "main.cpp"

print(f"Language : {LANGUAGE}")
print(f"Compile  : {' '.join(compile_command)}")
print(f"Run      : {' '.join(run_command)}")


Language : Rust
Compile  : C:\Users\HP\.cargo\bin\rustc.EXE main.rs -C opt-level=3 -C target-cpu=native -C codegen-units=1 -C lto=fat -C panic=abort -C strip=symbols -o main.exe
Run      : main.exe


In [ ]:
# ── PROMPT TEMPLATES ─────────────────────────────────────────────────────────
SYSTEM_PROMPT = f"""
Your task is to convert Python code into high-performance {LANGUAGE} code.
Respond only with {LANGUAGE} code. Do not explain; occasional code comments are fine.
The {LANGUAGE} code must produce identical output in the shortest possible time.
Do NOT wrap the output in markdown fences.
"""

def user_prompt_for(python: str) -> str:
    return f"""Port this Python to {LANGUAGE} for maximum runtime performance.
System information:
{system_info}

Your response will be written to {source_file} and compiled with:
{compile_command}

Respond only with {LANGUAGE} code.  Do NOT wrap in markdown fences.

Python code:

```python
{python}
```"""

def messages_for(python: str) -> list:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_prompt_for(python)},
    ]


In [ ]:
# ── CORE FUNCTIONS ───────────────────────────────────────────────────────────
def port(model: str, python: str) -> str:
    """Convert Python → target language via LLM; write to disk; return code."""
    client = clients[model]
    code = call_llm(client, model, messages_for(python))
    write_output(code, source_file)
    return code


def compile_and_run(code: str = "") -> str:
    """
    Compile the file on disk and run the binary.

    *code* is accepted but ignored — compile_and_run reads from source_file
    which was already written by port().  Signature kept for Gradio compat.
    """
    if LANGUAGE == "Rust":
        return compile_and_run_rust(compile_command, run_command)
    else:
        return compile_and_run_cpp(compile_command, run_command)


In [ ]:
# ── SAMPLE PYTHON (O(n²) maximum sub-array sum — intentionally slow) ─────────
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value

def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

n = 10000
initial_seed = 42
min_val = -10
max_val = 10

import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""


In [ ]:
# ── QUICK HEADLESS TEST (optional — skip for UI-only use) ────────────────────
# Converts the sample Python, writes the file, compiles, runs, prints output.

import time

print(f"=== Porting to {LANGUAGE} using {models[0]} ===")
t0 = time.time()
generated_code = port(models[0], python_hard)
t1 = time.time()
print(f"LLM call took {t1 - t0:.1f}s")
print("--- Generated code preview (first 10 lines) ---")
for line in generated_code.splitlines()[:10]:
    print(line)

print("\n--- Compilation & run ---")
output = compile_and_run()
print(output)


=== Porting to Rust using gemini-2.5-flash ===
LLM call took 43.6s
--- Generated code preview (first 10 lines) ---
use std::time::Instant;

// LCG (Linear Congruential Generator)
// The modulus `m` is implicitly 2^32, handled by `u32` wrapping arithmetic.
#[derive(Debug, Clone, Copy)]
struct Lcg {
    value: u32,
    a: u32,
    c: u32,
}

--- Compilation & run ---
❌ Compile/run error:
error: linker `link.exe` not found
  |
  = note: program not found

note: the msvc targets depend on the msvc linker but `link.exe` was not found

note: please ensure that Visual Studio 2017 or later, or Build Tools for Visual Studio were installed with the Visual C++ option

note: VS Code is a different product, and is not sufficient

error: aborting due to 1 previous error




In [ ]:
def convert_and_run(model: str, python_code: str):
    code = port(model, python_code)
    output = compile_and_run()
    return code, output


lang_label = LANGUAGE
lang_code  = "cpp" if LANGUAGE == "C++" else None   # "rust"/"go" unsupported by gr.Code

with gr.Blocks(theme=gr.themes.Soft(), css=CSS) as ui:
    gr.Markdown(f"# 🚀 Python → {lang_label} Code Converter (Day 5)")

    with gr.Row():
        with gr.Column(scale=1):
            python_input = gr.Code(
                label="Source: Python",
                language="python",
                lines=25,
                value=python_hard,
                elem_classes=["card"],
            )
        with gr.Column(scale=1):
            code_output = gr.Code(
                label=f"Output: {lang_label}",
                language=lang_code,
                lines=25,
                elem_classes=["card"],
            )

    with gr.Row(variant="panel"):
        with gr.Column(scale=4):
            model_select = gr.Dropdown(
                choices=models,
                label="Intelligence Engine",
                value=models[0],
            )
        with gr.Column(scale=1, min_width=140):
            convert_btn = gr.Button(
                "🔄 Convert", variant="primary", size="lg",
                elem_classes=["convert-btn"],
            )
        with gr.Column(scale=1, min_width=140):
            run_btn = gr.Button(
                f"▶ Run {lang_label}", variant="secondary", size="lg",
                elem_classes=["run-btn", "cpp"],
            )

    with gr.Row():
        with gr.Column(scale=1):
            python_out = gr.Textbox(
                label="Python Output",
                lines=4,
                interactive=False,
                elem_classes=["py-out"],
            )
        with gr.Column(scale=1):
            native_out = gr.Textbox(
                label=f"{lang_label} Output",
                lines=4,
                interactive=False,
                elem_classes=["cpp-out"],
            )

    convert_btn.click(
        fn=convert_and_run,
        inputs=[model_select, python_input],
        outputs=[code_output, native_out],
    )
    convert_btn.click(
        fn=run_python,
        inputs=[python_input],
        outputs=[python_out],
    )
    run_btn.click(
        fn=compile_and_run,
        inputs=[],
        outputs=[native_out],
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
